# Baselines - Part 1: segmentation heads, input pipeline, and smoke test

Sanity checks for the neural segmentation heads (`src/models/unet.py`), input standardisation and class weighting (`src/train/data.py`), and nested low-label subsets (`src/data/subsets.py`).

We verify parameter counts and step timing for both UNet widths, generate the fold I low-label splits, check per-channel statistics on training and validation patches, overfit a small batch of 8 patches, and benchmark a full epoch on a T4 GPU.

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0))
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
OUT = DRIVE / "results" / "baselines"
DEV = torch.device("cuda")

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy processed data to local disk
Copy the processed arrays from Drive to `/content/processed` once per session to avoid slow network reads during training.

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Parameter counts and step timing
Compare parameter counts for the two candidate UNet widths (`32-64-128` and `64-128-256`) across 128 (TESSERA), 64 (AlphaEarth), and 192 (concatenated) input channels, then time a mixed-precision step on a batch of 32 patches.

In [ ]:
from src.models.unet import build_head, count_parameters

HEADS = {"unet_32": {"widths": (32, 64, 128)}, "unet_64": {"widths": (64, 128, 256)}}
rows = [{"head": h, "in_channels": c, "parameters": count_parameters(build_head(h.split("_")[0], c, **kw))}
        for h, kw in HEADS.items() for c in (128, 64, 192)]
print(pd.DataFrame(rows).pivot(index="head", columns="in_channels", values="parameters").to_string())

x = torch.randn(32, 192, 128, 128, device=DEV)
y = torch.randint(0, 20, (32, 128, 128), device=DEV)  # 19 = ignored
for h in ("unet_32", "unet_64"):
    model = build_head("unet", 192, **HEADS[h]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    for step in range(6):
        if step == 1:  # the first step includes the cuDNN set-up
            torch.cuda.synchronize()
            t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = F.cross_entropy(model(x), y, ignore_index=19)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    torch.cuda.synchronize()
    print(f"{h}: {(time.time() - t0) / 5 * 1000:.0f} ms per step of 32 patches, "
          f"peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
del x, y, model, opt
torch.cuda.empty_cache()

## 5. Low-label subsets for fold I
For each seed, we build a nested ordering of the 1,455 training patches. We first sample a covering set over crop classes 1 to 18 from rarest to most common, picking one random training patch with at least 100 valid pixels for each uncovered class, and then shuffle the remaining patches. A subset at fraction `f` takes the first `round(f * 1455)` patches from this list, so subsets are nested within each seed and shared across all models. Validation (fold 4) and test (fold 5) splits remain unchanged.

In [ ]:
from src.data.splits import load_split
from src.data.subsets import label_subset

labels = np.load(PROC / "labels.npy")
valid = np.load(PROC / "valid_mask.npy")
part = load_split(PROC, "I")
train, val, test = (np.asarray(part[k]) for k in ("train", "val", "test"))
print(f"fold I: {len(train)} train, {len(val)} val, {len(test)} test patches")

FRACS, SEEDS = (0.01, 0.05, 0.10, 0.30, 1.0), (0, 1, 2)
subsets, rows = {}, []
for frac in FRACS:
    for seed in SEEDS:
        idx, info = label_subset(labels, valid, train, frac, seed)
        subsets[f"{round(frac * 100):03d}_s{seed}"] = {**info, "patches": idx.tolist()}
        rows.append({"label_frac": frac, "seed": seed, "patches": info["n_patches"], "covering_set": info["n_cover"],
                     "valid_pixels": sum(info["class_pixels"]), "missing_classes": info["missing_classes"]})
print(pd.DataFrame(rows).to_string(index=False))
for seed in SEEDS:
    sets = [set(subsets[f"{round(f * 100):03d}_s{seed}"]["patches"]) for f in FRACS]
    assert all(a <= b for a, b in zip(sets, sets[1:])), f"seed {seed}: subsets are not nested"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "label_subsets_foldI.json").write_text(json.dumps(subsets))
print("saved", OUT / "label_subsets_foldI.json")

## 6. Input standardisation and class weights
Per-channel mean and standard deviation are fitted on valid pixels from 400 random training patches. On those 400 patches, standardised channels have zero mean and unit variance. On validation patches, channel statistics shift slightly across Sentinel-2 tiles because the underlying regional distributions differ. We also compute the `1 / sqrt(frequency)` class weights on the training split.

In [ ]:
from src.eval.metrics import CLASS_NAMES
from src.train.data import IGNORE, InputStack, PatchDataset, class_weights


def channel_stats(stack, patches, batch_size=16):
    """Mean and std of every standardised channel over the valid pixels of `patches`."""
    loader = DataLoader(PatchDataset(stack, labels, valid, patches), batch_size=batch_size, num_workers=2)
    total = torch.zeros(stack.in_channels, dtype=torch.float64, device=DEV)
    total_sq, n, finite = torch.zeros_like(total), 0, True
    for raw, target, _ in loader:
        x = stack.decode(raw, DEV).double()
        finite &= bool(torch.isfinite(x).all())
        v = x.permute(1, 0, 2, 3)[:, target.to(DEV) != IGNORE]  # (C, valid pixels)
        total += v.sum(1)
        total_sq += (v * v).sum(1)
        n += v.shape[1]
    mean = total / n
    return mean, (total_sq / n - mean ** 2).clamp_min(0).sqrt(), finite


def describe(stack, mean, std):
    start = 0
    for name, c in zip(stack.sources, stack.channels):
        m, s = mean[start:start + c], std[start:start + c]
        print(f"  {name}: |mean| median {m.abs().median():.3f}, max {m.abs().max():.3f}; "
              f"std median {s.median():.3f}, range {s.min():.3f}-{s.max():.3f}")
        start += c


t0 = time.time()
stack = InputStack(PROC, ["tessera_2019", "ae_2019"]).fit_stats(train, valid, seed=0)
print(f"{stack.in_channels} channels {stack.channels}; statistics fitted on {len(stack.fit_patches)} "
      f"training patches in {time.time() - t0:.0f} s")

mean, std, finite = channel_stats(stack, stack.fit_patches)
print("same training patches (must be mean 0, std 1):")
describe(stack, mean, std)
assert finite and mean.abs().max() < 1e-3 and (std - 1).abs().max() < 1e-3, "standardisation is wrong"

tiles = pd.read_csv(PROC / "meta.csv").TILE.to_numpy()
rng = np.random.default_rng(0)
for tile in sorted(set(tiles[val])):
    sub = val[tiles[val] == tile]
    pick = np.sort(rng.choice(sub, min(32, len(sub)), replace=False))
    mean, std, finite = channel_stats(stack, pick)
    print(f"validation, {tile} ({len(pick)} of {len(sub)} patches):")
    describe(stack, mean, std)
    assert finite, "NaN or inf in the standardised inputs"

weights, counts = class_weights(labels, valid, train)
print(pd.DataFrame({"class": CLASS_NAMES, "pixels": counts, "share_%": 100 * counts / counts.sum(),
                    "weight": weights}).round(3).to_string())

## 7. Overfit test on 8 patches
Train the `32-64-128` UNet for 400 steps (`lr=1e-3`, mixed precision) on a single batch of 8 training patches to verify that the input decoding, target masking, and weighted cross-entropy loss work together as expected. The loss should drop below 20% of its initial value with overall accuracy above 0.9.

In [ ]:
from src.eval.metrics import confusion_matrix, summarize

torch.manual_seed(0)
few = np.sort(np.random.default_rng(0).choice(train, 8, replace=False))
raw, target, _ = next(iter(DataLoader(PatchDataset(stack, labels, valid, few), batch_size=8)))
x, y = stack.decode(raw, DEV), target.to(DEV)
model = build_head("unet", stack.in_channels).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")
w = torch.tensor(weights, dtype=torch.float32, device=DEV)
losses = []
model.train()
for step in range(400):
    with torch.autocast("cuda", dtype=torch.float16):
        loss = F.cross_entropy(model(x), y, weight=w, ignore_index=IGNORE)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    losses.append(loss.item())
    if step % 50 == 0 or step == 399:
        print(f"step {step:3d}  loss {losses[-1]:.4f}")
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    pred = model(x).argmax(1).cpu().numpy()
t = target.numpy()
s = summarize(confusion_matrix(t, pred, valid=t != IGNORE))
print(f"patches {few.tolist()}: {int((t != IGNORE).sum()):,} valid pixels, {int(np.isfinite(s['iou']).sum())} classes")
print(f"after 400 steps: loss {losses[-1]:.4f} ({losses[-1] / losses[0]:.1%} of the first), "
      f"OA {s['oa']:.4f}, mIoU {s['miou']:.4f}")
assert losses[-1] < 0.2 * losses[0] and s["oa"] > 0.9, "the UNet does not fit 8 patches: check inputs and targets"

## 8. Epoch timing benchmark
Measure the wall time of two training epochs over all 1,455 training patches (`32-64-128` UNet, batch size 32) for TESSERA alone and TESSERA + AlphaEarth. The first epoch includes disk reads; the second reflects cached throughput.

In [ ]:
def timed_epochs(sources, epochs=2, batch_size=32, workers=min(4, os.cpu_count())):
    st = InputStack(PROC, sources).fit_stats(train, valid, max_patches=50)
    loader = DataLoader(PatchDataset(st, labels, valid, train), batch_size=batch_size, shuffle=True,
                        num_workers=workers, pin_memory=True, drop_last=True, persistent_workers=True)
    model = build_head("unet", st.in_channels).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    w = torch.tensor(weights, dtype=torch.float32, device=DEV)
    for epoch in range(epochs):
        t0 = time.time()
        for raw, target, _ in loader:
            x, y = st.decode(raw, DEV), target.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(x), y, weight=w, ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        torch.cuda.synchronize()
        print(f"{'+'.join(sources)}: epoch {epoch + 1}: {time.time() - t0:.0f} s "
              f"({workers} loader workers), last loss {loss.item():.3f}")


for sources in (["tessera_2019"], ["tessera_2019", "ae_2019"]):
    timed_epochs(sources)